In [26]:
!pip install annoy

In [27]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from PIL import ImageFile
import pandas as pd
import numpy as np
import annoy
import os

In [28]:
base_model = ResNet50(weights="imagenet", include_top=False)
x = base_model.output
x = GlobalAveragePooling2D()(x)
model = Model(inputs=base_model.input, outputs=x)
mode=model.trainable = False

In [29]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [30]:
ImageFile.LOAD_TRUNCATED_IMAGES = True
trainPath='/content/gdrive/MyDrive/train'
valiPath='/content/gdrive/MyDrive/val'

In [31]:
trainImage = ImageDataGenerator(rescale=1./255).flow_from_directory(trainPath,target_size=(224,224),batch_size=32,class_mode='categorical')
valiImage = ImageDataGenerator(rescale=1./255).flow_from_directory(valiPath,target_size=(224,224),batch_size=32,class_mode='categorical')

# datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
# trainImage = ImageDataGenerator(rescale=1./255).flow_from_directory(trainPath,target_size=(224,224),batch_size=32,class_mode='categorical')
# valiImage = ImageDataGenerator(rescale=1./255).flow_from_directory(valiPath,target_size=(224,224),batch_size=32,class_mode='categorical')

Found 2872 images belonging to 18 classes.
Found 761 images belonging to 18 classes.


In [32]:
train_features = model.predict(trainImage, verbose=1)
val_features = model.predict(valiImage, verbose=1)

train_filenames = trainImage.filenames
val_filenames = valiImage.filenames

df_train = pd.DataFrame(train_features)
df_val = pd.DataFrame(val_features)

df_train.insert(0, "filename", train_filenames)
df_val.insert(0, "filename", val_filenames)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


90/90 ━━━━━━━━━━━━━━━━━━━━ 596s 6s/step
24/24 ━━━━━━━━━━━━━━━━━━━━ 162s 6s/step


In [33]:
df_train.to_csv("train_vectors.csv", index=False)
df_val.to_csv("val_vectors.csv", index=False)

In [34]:
import joblib

filename = 'CNN.sav'
joblib.dump(model, filename)

['CNN.sav']